# Завдання
## Знайти та реалізувати оригінальне рішення - синтезу оптимальної структури нейромережі за з більше за один індикатор ефективності моделі та результатів обробки;

Dataset (застосунок): колонки: `manufacturer, model, year, price, km_age, date_added`.
Підхід: підготовка даних → інженерія ознак → побудова та валідація DNN → архітектурний пошук (Grid / Random search по простору конфігурацій) → multi-objective оцінка → вибір найкращої моделі → звіт.


In [ ]:
import os
import json
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from sklearn.model_selection import train_test_split
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.metrics import mean_squared_error, mean_absolute_error, r2_score

import tensorflow as tf
from tensorflow import keras
from tensorflow.keras import layers

import warnings
warnings.filterwarnings("ignore")

RND = 42
np.random.seed(RND)
tf.random.set_seed(RND)


# Load data


In [ ]:
def load_and_clean(path='data/ev_cars.xlsx', column_names=None):
    if column_names is None:
        column_names = ['manufacturer', 'model', 'year', 'price', 'km_age', 'date_added']
    df = pd.read_excel(path, names=None)
    # ensure required cols exist (try to coerce)
    df = df.rename(columns={c: c.strip() for c in df.columns})
    # keep only relevant columns if present
    for c in column_names:
        if c not in df.columns:
            raise ValueError(f"Missing column {c} in input file")
    df = df[column_names].copy()

    # types
    df['date_added'] = pd.to_datetime(df['date_added'], errors='coerce')
    df['year'] = pd.to_numeric(df['year'], errors='coerce').astype('Int64')

    # price cleanup
    df['price'] = df['price'].astype(str).str.replace('[$, ]', '', regex=True)
    df['price'] = df['price'].str.replace('\xa0', '', regex=False)
    df['price'] = df['price'].str.replace(' ', '', regex=False)
    df['price'] = pd.to_numeric(df['price'], errors='coerce')
    df['km_age'] = pd.to_numeric(df['km_age'], errors='coerce')

    # drop rows with no date or no price
    df = df.dropna(subset=['price','km_age','date_added','year']).reset_index(drop=True)
    return df


# path to xlsx
path = 'data/ev_cars.xlsx'
df = load_and_clean(path)

print("Rows:", len(df))
display(df.head())


## 2. Feature engineering
- Від `date_added` зробимо `month`, `year_added` (можуть бути корисні сезонні/трендові ознаки).
- Компоненти `manufacturer` і `model` — категоріальні. Ми зробимо One-Hot кодування для `manufacturer` (якщо категорій багато — можна замінити на target encoding; тут OHE для простоти).
- `km_age`, `year`, похідні: `vehicle_age = date_added.year - year`.


In [ ]:
df['month_added'] = df['date_added'].dt.month.astype('Int64')
df['year_added'] = df['date_added'].dt.year.astype('Int64')
df['vehicle_age'] = df['year_added'] - df['year']

# remove rows with negative vehicle_age or missing
df = df[df['vehicle_age'] >= 0].reset_index(drop=True)

# Choose features and target
CAT_FEATURES = ['manufacturer']         # one-hot (low cardinality assumed)
NUM_FEATURES = ['km_age', 'vehicle_age']
OTHER_FEATS = ['month_added']           # cyclical encoding considered below

# cyclical encoding for month
df['month_sin'] = np.sin(2*np.pi*df['month_added']/12)
df['month_cos'] = np.cos(2*np.pi*df['month_added']/12)
NUM_FEATURES_EXT = NUM_FEATURES + ['month_sin','month_cos']

TARGET = 'price'


## 3. Train/Val/Test split + preprocessing pipeline
- Ми застосуємо ColumnTransformer: OneHot для manufacturer, StandardScaler для чисел.
- Збережемо pipeline для подальшого використання.

In [ ]:
X = df[CAT_FEATURES + NUM_FEATURES_EXT]
y = df[TARGET].values.astype(float)

X_trainval, X_test, y_trainval, y_test = train_test_split(X, y, test_size=0.15, random_state=RND)
X_train, X_val, y_train, y_val = train_test_split(X_trainval, y_trainval, test_size=0.1764706, random_state=RND)
# note: final proportions ~70/15/15

print("Shapes:", X_train.shape, X_val.shape, X_test.shape)

# Preprocessing pipeline
ohe = OneHotEncoder(handle_unknown='ignore', sparse_output=False)
scaler = StandardScaler()

preprocessor = ColumnTransformer(transformers=[
    ('cat', ohe, CAT_FEATURES),
    ('num', scaler, NUM_FEATURES_EXT)
], remainder='drop')

# Fit preprocessor on train
preprocessor.fit(X_train)

# Transform datasets
X_train_p = preprocessor.transform(X_train)
X_val_p = preprocessor.transform(X_val)
X_test_p = preprocessor.transform(X_test)

print("Processed shapes:", X_train_p.shape, X_val_p.shape, X_test_p.shape)


## 4. Модель: DNN (Dense). Простий фабричний конструктор мережі
Ми реалізуємо функцію `build_model(config)` яка будує Keras-модель за параметрами:
- `n_layers`, `units`, `activation`, `dropout`, `lr` (learning rate), `batch_norm` (вкл/вимк).
 Keras model from spec)

In [ ]:
input_dim = X_train_p.shape[1]

def build_model(config):
    inputs = keras.Input(shape=(input_dim,))
    x = inputs
    for i in range(config['n_layers']):
        x = layers.Dense(config['units'][i], activation=None)(x)
        if config.get('batch_norm', False):
            x = layers.BatchNormalization()(x)
        act = config.get('activation', 'relu')
        x = layers.Activation(act)(x)
        if config.get('dropout', 0.0) > 0:
            x = layers.Dropout(config['dropout'])(x)
    outputs = layers.Dense(1, activation='linear')(x)
    model = keras.Model(inputs=inputs, outputs=outputs)
    optimizer = keras.optimizers.Adam(learning_rate=config.get('lr', 1e-3))
    model.compile(optimizer=optimizer, loss='mse', metrics=[keras.metrics.RootMeanSquaredError(name='rmse'), keras.metrics.MeanAbsoluteError(name='mae')])
    return model


## 5. Multi-objective optimization strategy (оригінальна частина)
Ідея: шукати модель, що мінімізує комбінацію двох (або трьох) показників.
Ми використаємо як індикатори: RMSE та MAE (в одиницях валюти) і додатково R² для інформативності.

Composite score = normalized_RMSE + normalized_MAE
Normalization: ділимо RMSE і MAE на відповідні baseline (наприклад, на RMSE та MAE простої моделі — `mean baseline`), щоб мати порівнянні одиниці. Менше — краще.

Після пошуку за простим сітковим простором (обмеженим), ми виберемо k-найкращих конфігурацій та детальніше їх перевіримо (довше тренування, регуляризація).


In [ ]:
y_train_mean = np.mean(y_train)
baseline_rmse = np.sqrt(mean_squared_error(y_val, np.repeat(y_train_mean, len(y_val))))
baseline_mae = np.mean(np.abs(y_val - y_train_mean))
print("Baseline RMSE (mean pred on val):", baseline_rmse, " MAE:", baseline_mae)


## 6. Hyperparameter search (малий grid для демо)
Щоб не займати занадто багато часу, оберемо невеликий набір комбінацій.
Параметри (приклад): кількість шарів 1–3, units per layer (32,64,128), dropout 0–0.3, lr 1e-3/5e-4, activation relu / swish, batch_norm on/off.

Для кожної конфігурації:
- тренуємо коротко (наприклад `epochs=50`, `early_stopping` з `patience=5`),
- оцінюємо на валідації RMSE, MAE, R²,
- рахуємо composite score = (RMSE / baseline_rmse) + (MAE / baseline_mae).


In [ ]:
from itertools import product
from math import ceil
import time

layer_choices = [1,2]
unit_choices = [32,64]
dropout_choices = [0.0, 0.2]
activation_choices = ['relu','swish']  # swish available as tf.nn.swish via activation string in TF2.6+
lr_choices = [1e-6, 5e-4]
batch_norm_choices = [False, True]

grid = []
for n_layers, units, dropout, act, lr, bn in product(layer_choices, unit_choices, dropout_choices, activation_choices, lr_choices, batch_norm_choices):
    units_list = [units]*n_layers
    cfg = {'n_layers': n_layers, 'units': units_list, 'dropout': dropout, 'activation': act, 'lr': lr, 'batch_norm': bn}
    grid.append(cfg)

print("Grid size:", len(grid))

results = []
ES = keras.callbacks.EarlyStopping(monitor='val_rmse', patience=7, mode='min', restore_best_weights=True, verbose=0)

for i, cfg in enumerate(grid):
    print(f"[{i+1}/{len(grid)}] cfg={cfg}")
    model = build_model(cfg)
    t0 = time.time()
    hist = model.fit(X_train_p, y_train, validation_data=(X_val_p, y_val), epochs=80, batch_size=64, callbacks=[ES], verbose=0)
    t1 = time.time()
    val_preds = model.predict(X_val_p).ravel()
    rm = np.sqrt(mean_squared_error(y_val, val_preds))
    ma = np.mean(np.abs(y_val - val_preds))
    r2 = r2_score(y_val, val_preds)
    composite = (rm / baseline_rmse) + (ma / baseline_mae)
    results.append({'cfg':cfg, 'rmse':rm, 'mae':ma, 'r2':r2, 'composite':composite, 'time_s':t1-t0})
    print(f"  rmse={rm:.2f}, mae={ma:.2f}, r2={r2:.3f}, comp={composite:.3f}, time={t1-t0:.1f}s")


## 7. Аналіз результатів grid search — вибір топ-моделей



In [ ]:
res_df = pd.DataFrame([{
    'cfg': json.dumps(r['cfg']),
    'rmse': r['rmse'],
    'mae': r['mae'],
    'r2': r['r2'],
    'composite': r['composite'],
    'time_s': r['time_s']
} for r in results])

res_df_sorted = res_df.sort_values('composite').reset_index(drop=True)
display(res_df_sorted.head(10))


## 8. Додатковий етап — тонка перевірка top-k (довше тренування) та ensemble-перевірка
Ми візьмемо топ-3 за composite і натренуємо їх на повному train+val (70%) і перевіримо на test (15%).


In [ ]:
top_k = 3
top_cfgs = [json.loads(res_df_sorted.loc[i,'cfg']) for i in range(min(top_k, len(res_df_sorted)))]
X_full_train = np.vstack([X_train_p, X_val_p])
y_full_train = np.concatenate([y_train, y_val])

final_results = []
for cfg in top_cfgs:
    print("Retraining cfg:", cfg)
    model = build_model(cfg)
    ES_final = keras.callbacks.EarlyStopping(monitor='val_loss', patience=10, restore_best_weights=True, verbose=0)
    hist = model.fit(X_full_train, y_full_train, validation_split=0.1, epochs=20, batch_size=64, callbacks=[ES_final], verbose=0)
    preds_test = model.predict(X_test_p).ravel()
    rm = np.sqrt(mean_squared_error(y_test, preds_test))
    ma = np.mean(np.abs(y_test - preds_test))
    r2 = r2_score(y_test, preds_test)
    final_results.append({'cfg':cfg,'rmse':rm,'mae':ma,'r2':r2})
    print(f" Test RMSE={rm:.2f}, MAE={ma:.2f}, R2={r2:.3f}")

final_df = pd.DataFrame(final_results).sort_values('rmse').reset_index(drop=True)
display(final_df)


# 9. Візуалізація найкращого результату
- Побудуємо scatter (true vs pred), розподіл залишків, часовий ряд (sample)

In [ ]:
best_cfg = final_df.loc[0,'cfg']
best_model_cfg = best_cfg
print("Best cfg:", best_model_cfg)
# retrain best model to get object ( new for plot)
best_model = build_model(best_model_cfg)
best_model.fit(X_full_train, y_full_train, validation_split=0.1, epochs=200, batch_size=64, callbacks=[keras.callbacks.EarlyStopping(monitor='val_loss',patience=10,restore_best_weights=True)], verbose=0)
pred_test = best_model.predict(X_test_p).ravel()

plt.figure(figsize=(12,4))
plt.subplot(1,2,1)
plt.scatter(y_test, pred_test, alpha=0.5, s=10)
plt.plot([y_test.min(), y_test.max()], [y_test.min(), y_test.max()], 'k--')
plt.xlabel('True price'); plt.ylabel('Predicted price'); plt.title('True vs Pred (test)')

plt.subplot(1,2,2)
resid = y_test - pred_test
plt.hist(resid, bins=50)
plt.title('Residuals distribution (test)')
plt.tight_layout()
plt.show()



### 4.5 Тестування та верифікація
- Перехресна валідація time-aware (TimeSeriesSplit) для MA/ARIMA; для DNN проведено валідацію на відкладеній вибірці (70/15/15).
- Переобучення контролюється `EarlyStopping`.
- Метрики: RMSE, MAE, R². Composite score застосовано для відбору моделей.



In [ ]:
import joblib
os.makedirs("outputs", exist_ok=True)
# save preprocessor
joblib.dump(preprocessor, "outputs/preprocessor.joblib")
# save best model weights and config
best_model.export("outputs/best_model_keras")
with open("outputs/best_cfg.json", "w") as f:
    json.dump(best_model_cfg, f)
# save metrics
final_df.to_csv("outputs/final_results.csv", index=False)
print("Saved outputs to outputs/")
